# Exploratory Data Analysis

EDA revealed several unusually strong relationships between features and the target variable. Most notably, customers with `contract_length="Monthly"` exhibited a churn rate of 100%. This suggests that the dataset may contain synthetic patterns, target leakage, or business-specific churn definitions that are not documented.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from churn_mlops.data import load_raw_data, validate_data
from churn_mlops.models import FeatureBuilder

pd.set_option('display.width', 200)

## Load and validate data

Data from https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset.

We renamed the "testing" data set to "inference", because we will use the "training" set for model training, cross-validation and testing. We will treat the  "inference" set as "unseen" data for model training and instead use it later for inference and monitoring including data/concept drift.

In [ ]:
# load raw data
df_train = load_raw_data("customer_churn_dataset-training.csv", index_col="customerid")
df_infer = load_raw_data("customer_churn_dataset-inference.csv", index_col="customerid")

# validate data contract/schema
df_train = validate_data(df_train)
df_infer = validate_data(df_infer)

# separate column names into numeric and categorical
cols_num = df_train.columns[df_train.dtypes != 'string'].to_list()
cols_cat = df_train.columns[df_train.dtypes == 'string'].to_list()

## Summary of data sets

Observations:
* Our function `load_raw_data` ensures that both dataframes have the same column data types.
* There are no missing values in the data.
* Difference in distribution of numerical variables (e.g. different mean and median), most distinct for
    * `Age`
    * `Support Calls`
    * `Payment Delay`
    * `Total Spend`
* Comparatively higher churn incidence in training data.

In [ ]:
print(df_train.info())

In [ ]:
print(df_infer.info())

In [ ]:
print(df_train.describe())

In [ ]:
print(df_infer.describe())

In [ ]:
# verify if there is any missing data
num_nulls_train = df_train.isna().sum()
num_nulls_infer = df_infer.isna().sum()
num_nulls = pd.concat([num_nulls_train,num_nulls_infer], axis='columns')
num_nulls.columns = ['training','inference']
print(num_nulls)

## Correlation

Observations:
* Very different picture comparing correlations for "training" and "inference" data sets.
* Hardly any correlation between dependent variables for "inference" set, as opposed to "training" set.
* Different correlation pattern with target variable "Churn", indicating concept drift.

In [ ]:
corr_train = df_train.corr(numeric_only=True)
corr_infer = df_infer.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_train,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True
)
plt.title("Training data: correlation of numeric variables")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_infer,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
)
plt.title("Inference data: correlation of numeric variables")
plt.show()

## Distribution of categorical variables

Observations:
* `Gender`:             Shift from "Male" to "Female"
* `Contract Length`:    Initially lower share of "Monthly" subscriptions
* `Subscription Type`:  Equal share of "Basic", "Standard" and "Premium"

In [ ]:
# compare shares of categorical variables
cat_share_train = df_train[cols_cat].value_counts(normalize=True).sort_index()
cat_share_infer = df_infer[cols_cat].value_counts(normalize=True).sort_index()
cat_share = pd.concat([cat_share_train,cat_share_infer], axis='columns')
cat_share.columns = ['training','inference']
print(cat_share.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=cat_share,
    x='training',
    y='inference',
    hue='contract_length',
    size='subscription_type',
    style='gender',
    ax=ax
    )
ax.plot(
    [0.03, 0.08],
    [0.03, 0.08],
    color="black",
    linestyle="--",
    label="equality"
)
ax.legend(
    fontsize=10,
    title_fontsize=12,
    bbox_to_anchor=(1.0, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

In [ ]:
for var in ["gender", "contract_length", "subscription_type"]:
    cat_share_train = df_train[var].value_counts(normalize=True).sort_index()
    cat_share_infer = df_infer[var].value_counts(normalize=True).sort_index()
    cat_share = pd.concat([cat_share_train,cat_share_infer], axis='columns')
    cat_share.columns = ['training','inference']

    ax = cat_share.T.plot(
        kind="bar",
        stacked=True,
        figsize=(8, 4)
    )
    ax.set_ylabel("Share")
    ax.legend(title=var, bbox_to_anchor=(1.01, 1))
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    print(cat_share.round(3))

## Distribution of numerical variables

Observations:
* The distribution of numerical variables differs between both datasets.
* Our sub-sample based pairs-plot (differentiating by churn) reveals somewhat artifical relation/distribution patterns.

In [ ]:
for var in ['age', 'tenure', 'usage_frequency', 'payment_delay', 'total_spend', 'last_interaction']:
    sns.kdeplot(df_train[var], fill=True, alpha=0.3, label="Training")
    sns.kdeplot(df_infer[var], fill=True, alpha=0.3, label="Inference")
    plt.legend()
    plt.show()

In [ ]:
sns.histplot(df_train["support_calls"], bins=11, alpha=0.3, label="Training")
sns.histplot(df_infer["support_calls"], bins=11, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
train_sample = df_train.sample(n=1000, random_state=42)
sns.pairplot(
    train_sample,
    vars=cols_num[:-1],
    hue="churn",
    plot_kws={"alpha": 0.2}
)
plt.show()

## Interaction of categorical and numerical variables

In [ ]:
sns.barplot(data=df_train, x='payment_delay', y='contract_length', hue='gender')
plt.show()

In [ ]:
sns.barplot(data=df_train, x='support_calls', y='contract_length', hue='gender')
plt.show()

In [ ]:
sns.barplot(data=df_train, x='last_interaction', y='contract_length', hue='gender')
plt.show()

In [ ]:
sns.lmplot(data=df_train, x='tenure', y='total_spend', col='contract_length', hue='gender', scatter=False, order=3)
plt.show()

In [ ]:
sns.lmplot(data=df_train, x='age', y='support_calls', col='contract_length', hue='gender', scatter=False, order=3)
plt.show()

## Relationship between churn and other variables

Observations:
* Churn incidence of 100% for
    * `Age` above 50,
    * (number of)* `Support Calls` of 6 and more, 
    * `Payment Delay` of more than 20 days*, 
    * `Total Spend` of 500 or less,
    * `Contract Length = "Monthly"`.
* Relatively higher churn incidence for
    * lower `Age` (below 50),
    * `Tenure` of 5 or less and between 12 and 24 (months)*,
    * `Usage Frequency` below 10 (days per month)*,
    * (number of)* `Support Calls` between 3 and 5,
    * `Last Interaction` more than 15 days* ago,
    * `Gender = "Female"`.

* `*` Note that the unit of variables is a context based guess, as the data set is lacking exact variable definitions.

In [ ]:
print(df_train.groupby("contract_length")["churn"].agg(["count", "mean"]))
print(df_infer.groupby("contract_length")["churn"].agg(["count", "mean"]))

In [ ]:
for var in ['age', 'gender', 'tenure', 'usage_frequency', 'support_calls', 'payment_delay', 'subscription_type', 'contract_length', 'last_interaction']:
    plt.figure(figsize=(12,5))
    df_train.groupby(var)["churn"].mean().plot(kind="bar")
    plt.title(f"Churn Incidence by {var}")
    plt.ylim([0,1])
    plt.show()

In [ ]:
var = "total_spend"
tmp = df_train.copy()
tmp[var + " binned"] = pd.cut(tmp[var], bins=range(100,1001,100), include_lowest=True, precision=0)

plt.figure(figsize=(12,5))
tmp.groupby(var + " binned")["churn"].mean().plot(kind="bar")
plt.title(f"Churn Incidence by {var}")
plt.ylim([0,1])
plt.show()

In [ ]:
sns.lmplot(data=df_train, x='tenure', y='churn', col='contract_length', hue='gender', scatter=False, order=2)
plt.show()

## Customer ID

Eventually, we were wondering, if customer IDs in both data sets correspond to one and the same customer (maybe at a later point in time).

Conclusion: Comparing variables by `CustomerID`, we are certain, that this is not the case (e.g. ~ 50% different gender).

In [ ]:
# identify overlapping records by index (CustomerID)
df_inner = df_train.merge(df_infer, how='inner', left_index=True, right_index=True, suffixes=('_trn','_tst'))
print(df_inner.shape)

In [ ]:
# compare colum values for records with equal index (CustomerID)
[(col,(df_inner[col+'_trn']==df_inner[col+'_tst']).sum()) for col in df_train.columns]

## Engineered features

In [ ]:
fe = FeatureBuilder()
df_train_fe = fe.fit_transform(df_train)
print(df_train_fe.info())

In [ ]:
for var in ['late_payer', 'inactive_customer', 'contract_commitment']:
    plt.figure(figsize=(12,5))
    df_train_fe.groupby(var)["churn"].mean().plot(kind="bar")
    plt.title(f"Churn Incidence by {var}")
    # plt.ylim([0,1])
    plt.show()

In [ ]:
tmp = df_train_fe.copy()
for var in ["tenure_rel2_commitment", "engagement", "avg_spend_per_year", "avg_s_calls_per_year"]:
    
    tmp[var + " binned"] = pd.qcut(tmp[var], q=10, precision=2, duplicates='drop')

    plt.figure(figsize=(12,5))
    tmp.groupby(var + " binned")["churn"].mean().plot(kind="bar")
    plt.title(f"Churn Incidence by {var}")
    # plt.ylim([0,1])
    plt.show()

In [ ]:
corr_train_fe = df_train_fe.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_train_fe,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True
)
plt.title("Training data: correlation of numeric variables")
plt.show()

## Dataset Limitations

Several features exhibit partial deterministic relationships with the target variable, including contract length, payment delay, support calls, and age. As a result, model performance on this dataset may not accurately reflect real-world churn prediction performance. Findings and model evaluations should therefore be interpreted with caution.